# Model A — Speech to Text

Transcribes patient audio (Kinyarwanda or English) using pretrained Whisper models.
No training data required — models are used as-is.

| Language | Model |
|---|---|
| Kinyarwanda | `akera/whisper-large-v3-kin-200h-v2` |
| English | `openai/whisper-large-v3` |

In [ ]:
!pip install -q -U transformers accelerate librosa soundfile langdetect

In [ ]:
import torch
import numpy as np
import librosa
from transformers import pipeline
from langdetect import detect, LangDetectException
from typing import Optional

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
SR     = 16_000

print(f"Device: {DEVICE}")

In [ ]:
kin_pipe = pipeline(
    "automatic-speech-recognition",
    model="akera/whisper-large-v3-kin-200h-v2",
    torch_dtype=DTYPE,
    device=DEVICE,
    return_timestamps=True,
)

eng_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    torch_dtype=DTYPE,
    device=DEVICE,
    return_timestamps=True,
)

print("Models loaded.")

In [ ]:
def detect_language(audio: np.ndarray, sr: int) -> str:
    result = eng_pipe(
        {"array": audio, "sampling_rate": sr},
        generate_kwargs={"task": "transcribe", "language": None},
        return_timestamps=False,
    )
    try:
        code = detect(result.get("text", ""))
        return "kinyarwanda" if code == "rw" else "english"
    except LangDetectException:
        return "kinyarwanda"


def compute_confidence(chunks: list) -> float:
    if not chunks:
        return 0.0
    bad_tags  = ["[BLANK_AUDIO]", "[MUSIC]"]
    penalties = sum(
        0.3 for c in chunks
        if any(tag in c.get("text", "") for tag in bad_tags)
    )
    return round(max(0.0, 1.0 - penalties / len(chunks)), 3)

In [ ]:
def transcribe_segment(audio: np.ndarray, sr: int, language: Optional[str] = None) -> dict:
    lang = language or detect_language(audio, sr)
    pipe = kin_pipe if lang == "kinyarwanda" else eng_pipe

    gen_kwargs = {"task": "transcribe"}
    if lang == "english":
        gen_kwargs["language"] = "english"

    result = pipe(
        {"array": audio, "sampling_rate": sr},
        generate_kwargs=gen_kwargs,
        return_timestamps=True,
    )
    chunks = result.get("chunks", [])

    return {
        "text":       result.get("text", "").strip(),
        "language":   lang,
        "confidence": compute_confidence(chunks),
    }


def transcribe_file(path: str, language: Optional[str] = None, segment_s: float = 30.0) -> dict:
    audio, _ = librosa.load(path, sr=SR, mono=True)
    seg_len  = int(segment_s * SR)
    segments = [
        transcribe_segment(audio[i : i + seg_len], SR, language)
        for i in range(0, len(audio), seg_len)
    ]

    lang_votes = [s["language"] for s in segments]

    return {
        "full_text":         " ".join(s["text"] for s in segments if s["text"]),
        "dominant_language": max(set(lang_votes), key=lang_votes.count),
        "mean_confidence":   round(float(np.mean([s["confidence"] for s in segments])), 3),
        "segments":          segments,
    }

---
## Usage

### Option A — Upload a file

In [ ]:
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

result = transcribe_file(filename)
print(result["full_text"])

### Option B — Pass a numpy array directly

In [ ]:
# audio_array = <np.ndarray at 16kHz>
# result = transcribe_segment(audio_array, SR, language="english")
# print(result["text"])